# GRM aux-LM regularization — controlled A/B (does it buy OOD generalization?)

Trains TWO matched **Qwen2.5-0.5B-Instruct** reward models on the SAME cleaned UltraFeedback (same 
seed/lr/epochs/effective-batch), differing only in the **GRM auxiliary LM loss** 
(`L = L_BT + alpha*L_LM`, arXiv:2406.10216):
- **base** : `train.aux_lm_coef=0` (plain Bradley-Terry)
- **grm**  : `train.aux_lm_coef=0.05` (keeps the LM head able to decode -> regularizes hidden states)

Then evals BOTH on **in-distribution** (cleaned UF held-out) and **OOD** (`Anthropic/hh-rlhf` test — a 
different distribution the RM never trained on). GRM claim: comparable in-dist, **higher OOD** (+3-8). 
~3-4 h on a T4. (GRM uses batch 2 / grad_accum 8 because the LM head makes [B,T,vocab] logits.)

In [ ]:
import torch, json
cap=torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
P100=tuple(cap)==(6,0); json.dump({'p100':P100},open('/tmp/gpu.json','w'))
print(f'GPU: {name} cap={cap} -> '+('P100 fp32' if P100 else 'T4 bf16'))

In [ ]:
!pip install -q 'transformers>=4.44,<5' 'datasets>=2.20' 'accelerate>=0.33' 'peft>=0.12' pytest
import json, subprocess, sys
if json.load(open('/tmp/gpu.json'))['p100']:
    subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.44,<4.48'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','torch==2.4.1','torchvision==0.19.1',
                    'torchaudio==2.4.1','--index-url','https://download.pytorch.org/whl/cu121'],check=True)

In [ ]:
import os
!rm -rf /kaggle/working/rlhf-pipeline && git clone -q https://github.com/TheYellowDuck/RLHF-pipeline.git /kaggle/working/rlhf-pipeline
%cd /kaggle/working/rlhf-pipeline

In [ ]:
!python scripts/smoke_test.py 2>&1 | tail -3   # exercises the aux_lm_check (GRM) path too

## Config

In [ ]:
import json
P100=json.load(open('/tmp/gpu.json'))['p100']
MODEL='Qwen/Qwen2.5-0.5B-Instruct'
DATA_CLEAN='argilla/ultrafeedback-binarized-preferences-cleaned'
DATA_OOD='Anthropic/hh-rlhf'      # true OOD: different distribution, never trained on
RM_SAMPLES=6000
DTYPE,BF16=('float32','false') if P100 else ('bfloat16','true')
print(f'model={MODEL} samples={RM_SAMPLES} dtype={DTYPE}')

## A) baseline RM — plain Bradley-Terry (aux_lm_coef=0)

In [ ]:
!python scripts/train_reward_model.py \
  -o model.name_or_path=$MODEL -o model.dtype=$DTYPE \
  -o data.name=$DATA_CLEAN -o data.train_split='train[2000:]' -o data.eval_split='train[:2000]' \
  -o data.max_samples=$RM_SAMPLES -o data.max_eval_samples=2000 -o data.max_length=512 \
  -o train.epochs=1 -o train.batch_size=4 -o train.grad_accum=4 -o train.bf16=$BF16 \
  -o train.lr=1.0e-5 -o train.label_smoothing=0.0 -o data.max_pair_similarity=1.0 \
  -o output_dir=checkpoints/rm_base

## B) GRM RM — same recipe + aux-LM loss (aux_lm_coef=0.05, batch 2/grad_accum 8)

In [ ]:
!python scripts/train_reward_model.py \
  -o model.name_or_path=$MODEL -o model.dtype=$DTYPE \
  -o data.name=$DATA_CLEAN -o data.train_split='train[2000:]' -o data.eval_split='train[:2000]' \
  -o data.max_samples=$RM_SAMPLES -o data.max_eval_samples=2000 -o data.max_length=512 \
  -o train.epochs=1 -o train.batch_size=2 -o train.grad_accum=8 -o train.bf16=$BF16 \
  -o train.lr=1.0e-5 -o train.label_smoothing=0.0 -o data.max_pair_similarity=1.0 \
  -o train.aux_lm_coef=0.05 \
  -o output_dir=checkpoints/rm_grm

## Evaluate both — in-distribution (cleaned UF) vs OOD (HH-RLHF)

In [ ]:
import subprocess
def acc(ckpt, data, split):
    c=(f"python scripts/evaluate.py rm-accuracy --reward-model {ckpt} "
       f"--data {data} --split '{split}' --max-samples 2000 --batch-size 8")
    print('$',c,flush=True); o=subprocess.run(c,shell=True,capture_output=True,text=True)
    print(o.stdout[-300:]);
    if o.returncode: print(o.stderr[-800:])
    return o.stdout.strip().splitlines()[-1] if o.stdout.strip() else o.stderr[-200:]
b_id=acc('checkpoints/rm_base', DATA_CLEAN, 'train[:2000]')
g_id=acc('checkpoints/rm_grm',  DATA_CLEAN, 'train[:2000]')
b_ood=acc('checkpoints/rm_base', DATA_OOD, 'test')
g_ood=acc('checkpoints/rm_grm',  DATA_OOD, 'test')
md=('# GRM aux-LM A/B results (Qwen2.5-0.5B-Instruct)\n\n'
    f'- train: `{DATA_CLEAN}` ({RM_SAMPLES} pairs), 1 epoch, lr 1e-5, only difference = aux_lm_coef\n\n'
    '| RM | in-dist (cleaned UF held-out) | OOD (HH-RLHF test) |\n|---|---|---|\n'
    f'| base (BT only) | {b_id} | {b_ood} |\n'
    f'| GRM (aux 0.05) | {g_id} | {g_ood} |\n\n'
    'GRM works if OOD(GRM) > OOD(base) while in-dist stays comparable.\n')
open('/kaggle/working/RESULTS.md','w').write(md); print('\n'+md)

### Read
If the GRM row beats base on **OOD** with similar in-dist, the aux-LM regularizer generalizes better 
(arXiv:2406.10216) — and it's worth re-running at 1.5B to lift the 0.8025 RM. If not, the lever 
doesn't help on this data/scale and the 0.8025 BT RM stays the best.